In [0]:

path = "/Volumes/dbw_agricoop_poc/ingestion/ml_inputs/"

# Dataset d'entraînement
df_train = spark.read.parquet(
    path + "ventes_hebdo.parquet"
)

# Dataset réservé à l'évaluation
df_latente = spark.read.parquet(
    path + "demande_latente_hebdo.parquet"
)

print("Données ML :", df_train.count())
print("Données latentes :", df_latente.count())

df_train.printSchema()
display(df_train.limit(5))


In [0]:

from pyspark.sql import functions as F
import pandas as pd

# Conversion du DataFrame Spark en pandas
ventes = df_train.toPandas()

# 1. Dimensions du dataset
print("Nombre de lignes :", df_train.count())
print("Nombre de colonnes :", len(df_train.columns))

# 2. Liste des variables et leurs types
df_train.printSchema()

# 3. Vérifier la couverture temporelle
df_train.agg(
    F.countDistinct("semaine").alias("nb_semaines"),
    F.countDistinct("code").alias("nb_produits"),
    F.min("semaine").alias("debut"),
    F.max("semaine").alias("fin")
).show(truncate=False)

# 4. Détecter les valeurs manquantes
nulls = df_train.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_train.columns
]).first().asDict()

print("Valeurs manquantes :")
print({k: v for k, v in nulls.items() if v > 0})


In [0]:

# Cellule 2 — Même configuration que le modèle local

df = ventes.copy()
df["semaine"] = pd.to_datetime(df["semaine"]).dt.normalize()
df = df.sort_values(["code", "semaine"]).reset_index(drop=True)

for col in ("code", "categorie"):
    df[col] = df[col].astype("category")

# Variables du modèle local
FEATURES = [
    "code", "categorie", "semaine_annee", "mois",
    "jours_ramadan", "jours_avant_ramadan",
    "jours_avant_aid", "stock_debut",
    "ventes_lag1", "ventes_lag2", "ventes_lag4",
    "ventes_lag52", "ventes_moy4", "rupture_sem_prec"
]

# Modèle corrigé : sans variables liées au stock
FEATURES_DEMANDE = [
    f for f in FEATURES
    if f not in ("stock_debut", "rupture_sem_prec")
]

assert set(FEATURES).issubset(df.columns)
assert "ventes" not in FEATURES_DEMANDE
assert "jours_rupture" not in FEATURES_DEMANDE

# Entraînement : deux campagnes
# Test : troisième campagne
DEBUT_TEST = pd.Timestamp("2025-09-29")

train = df[df["semaine"] < DEBUT_TEST].copy()
test = df[df["semaine"] >= DEBUT_TEST].copy()

assert (len(train), len(test)) == (1144, 572)

print("Variables ML :", len(FEATURES_DEMANDE))
print("Train :", len(train), "lignes")
print("Test :", len(test), "lignes")
print("Semaines Train :", train["semaine"].nunique())
print("Semaines Test :", test["semaine"].nunique())

print("Ruptures Train :", round((train["jours_rupture"] > 0).mean() * 100, 1), "%")
print("Ruptures Test :", round((test["jours_rupture"] > 0).mean() * 100, 1), "%")


In [0]:

%pip install lightgbm


In [0]:

# Cellule 3 — Préparation de l'entraînement

import lightgbm as lgb
import mlflow
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Préparer les observations sans rupture pour le modèle corrigé
train_sans_rupture = train[
    train["jours_rupture"] == 0
].copy()

# Préparer les features et la cible
X_train = train[FEATURES_DEMANDE]
y_train = train["ventes"]

X_train_corrige = train_sans_rupture[FEATURES_DEMANDE]
y_train_corrige = train_sans_rupture["ventes"]

X_test = test[FEATURES_DEMANDE]
y_test = test["ventes"]

print("LightGBM :", lgb.__version__)
print("MLflow :", mlflow.__version__)
print("Train complet :", len(X_train))
print("Train sans rupture :", len(X_train_corrige))
print("Test :", len(X_test))
print("Features :", FEATURES_DEMANDE)


In [0]:

# Cellule 4 — Expérience MLflow Databricks

import mlflow

EXPERIMENT_NAME = "/Shared/agricoop-prevision-demande"

mlflow.set_experiment(EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("Expérience :", experiment.name)
print("Experiment ID :", experiment.experiment_id)
print("Tracking URI :", mlflow.get_tracking_uri())


In [0]:

# Cellule 5 — Demande latente : évaluation uniquement

latente = df_latente.toPandas()

print("Colonnes latentes :", latente.columns.tolist())

assert {"semaine", "code", "demande_latente"}.issubset(latente.columns)

latente["semaine"] = pd.to_datetime(
    latente["semaine"]
).dt.normalize()

latente["code"] = latente["code"].astype(str)

# Jointure sur les 52 semaines de test
test = test.drop(
    columns="demande_latente", errors="ignore"
).merge(
    latente[["semaine", "code", "demande_latente"]],
    on=["semaine", "code"],
    how="left",
    validate="one_to_one"
)

assert len(test) == 572
assert test["demande_latente"].notna().all()
assert "demande_latente" not in FEATURES_DEMANDE

avec_rupture = test["jours_rupture"] > 0

print("Observations test :", len(test))
print("Observations en rupture :", avec_rupture.sum())
print("Demande latente manquante :", test["demande_latente"].isna().sum())
print("Validation demande latente : OK")


In [0]:

# ============================================================
# CELLULE 3 — BASELINE, LIGHTGBM NAÏF ET LIGHTGBM CORRIGÉ
# Évaluation sur la demande réelle + suivi MLflow Databricks
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import mlflow

# ------------------------------------------------------------
# 1. Joindre la demande latente au TEST uniquement
# ------------------------------------------------------------

lat = (
    latente.copy()
    if isinstance(latente, pd.DataFrame)
    else latente.toPandas()
)

lat["semaine"] = pd.to_datetime(lat["semaine"]).dt.normalize()
lat["code_str"] = lat["code"].astype(str)

# Supprimer les anciennes colonnes si la cellule est réexécutée
test = test.drop(
    columns=[
        "demande_latente", "code_str",
        "pred_baseline", "pred_naif", "pred_corrige"
    ],
    errors="ignore"
)

test["code_str"] = test["code"].astype(str)

test = (
    test.merge(
        lat[["semaine", "code_str", "demande_latente"]],
        on=["semaine", "code_str"],
        how="left",
        validate="one_to_one"
    )
    .drop(columns="code_str")
)

assert len(test) == 572
assert test["demande_latente"].notna().all()
assert "demande_latente" not in FEATURES + FEATURES_DEMANDE

# ------------------------------------------------------------
# 2. CORRECTION : aligner les catégories pandas Train/Test
# ------------------------------------------------------------

for col in ["code", "categorie"]:

    train[col] = train[col].astype("category")

    categories_train = train[col].cat.categories

    test[col] = pd.Categorical(
        test[col].astype(str),
        categories=categories_train,
        ordered=train[col].cat.ordered
    )

    assert test[col].notna().all(), (
        f"Catégories inconnues dans test[{col}]"
    )

    assert list(train[col].cat.categories) == list(
        test[col].cat.categories
    )

print("Catégories Train/Test alignées : OK")

# Observations ayant subi une rupture de stock
avec_rupture = test["jours_rupture"] > 0

# ------------------------------------------------------------
# 3. Métrique WAPE
# ------------------------------------------------------------

def wape(reel, prevu):
    """Somme des erreurs absolues / somme des valeurs réelles."""
    reel = np.asarray(reel)
    prevu = np.asarray(prevu)

    return np.abs(reel - prevu).sum() / reel.sum()

# ------------------------------------------------------------
# 4. Hyperparamètres identiques au notebook local
# ------------------------------------------------------------

PARAMS = dict(
    objective="poisson",
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=10,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

# ------------------------------------------------------------
# 5. MODÈLE 1 — Baseline : même semaine l'année précédente
# ------------------------------------------------------------

test["pred_baseline"] = test["ventes_lag52"].fillna(0)

# ------------------------------------------------------------
# 6. MODÈLE 2 — LightGBM naïf
# Toutes les observations, y compris les semaines en rupture
# ------------------------------------------------------------

modele_naif = lgb.LGBMRegressor(**PARAMS)

modele_naif.fit(
    train[FEATURES],
    train["ventes"]
)

test["pred_naif"] = modele_naif.predict(
    test[FEATURES]
)

# ------------------------------------------------------------
# 7. MODÈLE 3 — LightGBM corrigé
# Entraînement sans rupture et sans variables de stock
# ------------------------------------------------------------

sans_rupture = train["jours_rupture"] == 0

modele_corrige = lgb.LGBMRegressor(**PARAMS)

modele_corrige.fit(
    train.loc[sans_rupture, FEATURES_DEMANDE],
    train.loc[sans_rupture, "ventes"]
)

test["pred_corrige"] = modele_corrige.predict(
    test[FEATURES_DEMANDE]
)

# ------------------------------------------------------------
# 8. Calcul des métriques
# ------------------------------------------------------------

def metriques(col):

    return {
        "wape_ventes": wape(
            test["ventes"],
            test[col]
        ),

        "wape_demande": wape(
            test["demande_latente"],
            test[col]
        ),

        "wape_demande_ruptures": wape(
            test.loc[avec_rupture, "demande_latente"],
            test.loc[avec_rupture, col]
        ),

        "biais_demande": (
            test[col].sum()
            / test["demande_latente"].sum()
            - 1
        )
    }

# ------------------------------------------------------------
# 9. Comparaison avec les résultats locaux
# ------------------------------------------------------------

MODELES = [
    ("Baseline (an dernier)", "pred_baseline", []),
    ("LightGBM naïf", "pred_naif", FEATURES),
    ("LightGBM corrigé", "pred_corrige", FEATURES_DEMANDE)
]

LOCAL = {
    "Baseline (an dernier)": (0.603, 0.685),
    "LightGBM naïf": (0.478, 0.895),
    "LightGBM corrigé": (0.464, 0.544)
}

# ------------------------------------------------------------
# 10. Suivi des trois modèles avec MLflow
# ------------------------------------------------------------

lignes = {}

for nom, col, feats in MODELES:

    m = metriques(col)

    with mlflow.start_run(run_name=f"{nom} — Azure"):

        mlflow.set_tags({
            "plateforme": "azure-databricks",
            "donnees": "silver/ml/inputs",
            "donnees.synthetiques": "oui"
        })

        mlflow.log_params({
            "debut_test": str(DEBUT_TEST.date()),
            "features": ",".join(feats) or "lag52",
            **(PARAMS if feats else {})
        })

        mlflow.log_metrics({
            k: float(v)
            for k, v in m.items()
        })

    lignes[nom] = {
        "WAPE demande (Azure)": m["wape_demande"],
        "WAPE demande (local)": LOCAL[nom][0],
        "WAPE ruptures (Azure)": m["wape_demande_ruptures"],
        "WAPE ruptures (local)": LOCAL[nom][1],
        "Biais (Azure)": m["biais_demande"]
    }

# ------------------------------------------------------------
# 11. Affichage des résultats
# ------------------------------------------------------------

print(
    f"Observations produit-semaine en rupture : "
    f"{avec_rupture.sum()} sur {len(test)}"
)

resultats_azure = pd.DataFrame(lignes).T

print("\nCOMPARAISON AZURE VS LOCAL (%)")
print(
    (resultats_azure * 100)
    .round(1)
    .to_string()
)

print("\nValidation cellule 3 : OK")


In [0]:

# ============================================================
# CELLULE 4 — WALK-FORWARD + MODÈLE DE PRODUCTION
# LightGBM corrigé avec suivi MLflow Databricks
# ============================================================

import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.lightgbm

from mlflow.models import infer_signature

PAS = 4  # Réentraînement toutes les 4 semaines

# ------------------------------------------------------------
# 1. WALK-FORWARD SUR LES 52 SEMAINES DE TEST
# ------------------------------------------------------------

semaines_test = sorted(test["semaine"].unique())
previsions = []

for i in range(0, len(semaines_test), PAS):

    bloc = semaines_test[i:i + PAS]

    # Uniquement les semaines passées sans rupture
    historique = df[
        (df["semaine"] < bloc[0])
        & (df["jours_rupture"] == 0)
    ].copy()

    # Entraîner avec les paramètres du modèle local
    modele = lgb.LGBMRegressor(**PARAMS)

    modele.fit(
        historique[FEATURES_DEMANDE],
        historique["ventes"]
    )

    # Préparer les semaines à évaluer
    a_prevoir = df[
        df["semaine"].isin(bloc)
    ].copy()

    X_prevoir = a_prevoir[FEATURES_DEMANDE].copy()

    # Garantir la compatibilité des variables catégorielles
    for col in ("code", "categorie"):

        X_prevoir[col] = pd.Categorical(
            X_prevoir[col].astype(str),
            categories=historique[col].cat.categories,
            ordered=historique[col].cat.ordered
        )

        assert X_prevoir[col].notna().all()

    predictions = modele.predict(X_prevoir)

    previsions.append(
        pd.DataFrame({
            "semaine": a_prevoir["semaine"].to_numpy(),
            "code_str": a_prevoir["code"].astype(str).to_numpy(),
            "pred_walk_forward": predictions
        })
    )

# ------------------------------------------------------------
# 2. RATTACHER LES PRÉDICTIONS AU TEST
# ------------------------------------------------------------

pred_wf = pd.concat(previsions, ignore_index=True)

assert len(previsions) == 13
assert len(pred_wf) == 572

test = (
    test.drop(
        columns=["pred_walk_forward", "code_str"],
        errors="ignore"
    )
    .assign(code_str=lambda x: x["code"].astype(str))
    .merge(
        pred_wf,
        on=["semaine", "code_str"],
        how="left",
        validate="one_to_one"
    )
    .drop(columns="code_str")
)

assert test["pred_walk_forward"].notna().all()

# ------------------------------------------------------------
# 3. ÉVALUATION DU WALK-FORWARD
# ------------------------------------------------------------

avec_rupture = test["jours_rupture"] > 0

m = metriques("pred_walk_forward")

with mlflow.start_run(
    run_name="LightGBM corrigé, walk-forward — Azure"
):

    mlflow.set_tags({
        "plateforme": "azure-databricks",
        "donnees": "silver/ml/inputs",
        "donnees.synthetiques": "oui",
        "validation": "walk-forward"
    })

    mlflow.log_params({
        **PARAMS,
        "debut_test": str(DEBUT_TEST.date()),
        "features": ",".join(FEATURES_DEMANDE),
        "pas_reentrainement_semaines": PAS,
        "nombre_reentrainements": len(previsions)
    })

    mlflow.log_metrics({
        k: float(v)
        for k, v in m.items()
    })

# ------------------------------------------------------------
# 4. MODÈLE FINAL — TOUT L'HISTORIQUE SANS RUPTURE
# ------------------------------------------------------------

historique = df[
    df["jours_rupture"] == 0
].copy()

modele_prod = lgb.LGBMRegressor(**PARAMS)

modele_prod.fit(
    historique[FEATURES_DEMANDE],
    historique["ventes"]
)

# ------------------------------------------------------------
# 5. SIGNATURE ET EXEMPLE D'ENTRÉE POUR UNITY CATALOG
# ------------------------------------------------------------

# 12 variables explicatives, dans le bon ordre
X_exemple = historique[
    FEATURES_DEMANDE
].head(5).copy()

# Inférer le contrat entrée/sortie du modèle
signature = infer_signature(
    X_exemple,
    modele_prod.predict(X_exemple)
)

print("Signature du modèle :")
print(signature)

# ------------------------------------------------------------
# 6. ENREGISTRER LE MODÈLE FINAL DANS MLFLOW
# ------------------------------------------------------------

with mlflow.start_run(
    run_name="Production : LightGBM corrigé — Azure"
) as run:

    mlflow.set_tags({
        "plateforme": "azure-databricks",
        "role": "production-candidate",
        "donnees.synthetiques": "oui"
    })

    mlflow.log_params({
        **PARAMS,
        "features": ",".join(FEATURES_DEMANDE),
        "semaines_entrainement": int(
            historique["semaine"].nunique()
        ),
        "observations_entrainement": len(historique),
        "exclusion_ruptures": True
    })

    # Enregistrer le modèle avec son contrat d'entrée/sortie
    model_info = mlflow.lightgbm.log_model(
        lgb_model=modele_prod,
        name="modele",
        input_example=X_exemple,
        signature=signature
    )

    RUN_PROD = run.info.run_id
    MODEL_URI = model_info.model_uri

# ------------------------------------------------------------
# 7. AFFICHER LES RÉSULTATS
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RÉSULTATS WALK-FORWARD")
print("=" * 55)

print(f"Réentraînements : {len(previsions)}")

print(
    f"WAPE demande réelle : {m['wape_demande']:.1%}"
    f" (local : 45,9 %)"
)

print(
    f"WAPE en rupture : {m['wape_demande_ruptures']:.1%}"
    f" (local : 51,2 %)"
)

print(
    f"Biais : {m['biais_demande']:.1%}"
    f" (local : -16,9 %)"
)

print("\n" + "=" * 55)
print("MODÈLE FINAL MLFLOW")
print("=" * 55)

print("Run ID :", RUN_PROD)
print("Model URI :", MODEL_URI)
print("Observations d'entraînement :", len(historique))
print("Nombre de features :", len(FEATURES_DEMANDE))

print("\nValidation cellule 4 : OK")


In [0]:

# CELLULE 5 — Modèle MLflow avec prétraitement d'inférence

import pandas as pd
import numpy as np
import mlflow
import mlflow.pyfunc
import lightgbm as lgb
import sklearn
import cloudpickle

from mlflow.models import infer_signature

CATEGORIELLES = ["code", "categorie"]
NUMERIQUES = [
    c for c in FEATURES_DEMANDE
    if c not in CATEGORIELLES
]

class AgriCoopPrevision(mlflow.pyfunc.PythonModel):

    def __init__(self, modele, features, categories):
        self.modele = modele
        self.features = list(features)
        self.categories = categories

    def predict(self, context, model_input, params=None):
        X = pd.DataFrame(model_input)[self.features].copy()

        # Restaurer les catégories attendues par LightGBM
        for col in CATEGORIELLES:
            valeurs = X[col].astype(str)

            inconnues = (
                set(valeurs.unique())
                - set(self.categories[col])
            )

            if inconnues:
                raise ValueError(
                    f"Valeurs inconnues pour {col}: {inconnues}"
                )

            X[col] = pd.Categorical(
                valeurs,
                categories=self.categories[col]
            )

        # Schéma d'entrée stable pour MLflow
        for col in NUMERIQUES:
            X[col] = pd.to_numeric(
                X[col], errors="raise"
            ).astype("float64")

        return self.modele.predict(X)


# Conserver les catégories apprises
categories_modele = {
    col: historique[col].cat.categories.tolist()
    for col in CATEGORIELLES
}

wrapper = AgriCoopPrevision(
    modele=modele_prod,
    features=FEATURES_DEMANDE,
    categories=categories_modele
)

# Exemple d'entrée de type API : chaînes + nombres
X_api = test[FEATURES_DEMANDE].tail(10).copy()

for col in CATEGORIELLES:
    X_api[col] = X_api[col].astype(str)

for col in NUMERIQUES:
    X_api[col] = X_api[col].astype("float64")

# Vérifier avant enregistrement
pred_wrapper = wrapper.predict(None, X_api)

X_reference = X_api.copy()
for col in CATEGORIELLES:
    X_reference[col] = pd.Categorical(
        X_reference[col],
        categories=categories_modele[col]
    )

pred_reference = modele_prod.predict(X_reference)

np.testing.assert_allclose(
    pred_wrapper,
    pred_reference,
    rtol=1e-8,
    atol=1e-8
)

print("Prétraitement catégoriel : OK")

# Signature compatible avec les données envoyées par API
signature_api = infer_signature(
    X_api,
    pred_wrapper
)

# Enregistrer un NOUVEAU modèle, sans réentraînement
with mlflow.start_run(
    run_name="Production candidate — inférence corrigée"
) as run:

    mlflow.set_tags({
        "plateforme": "azure-databricks",
        "role": "production-candidate",
        "donnees.synthetiques": "oui",
        "inference": "categories-restaurees"
    })

    info = mlflow.pyfunc.log_model(
        name="modele_inference",
        python_model=wrapper,
        input_example=X_api.head(5),
        signature=signature_api,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"lightgbm=={lgb.__version__}",
            f"pandas=={pd.__version__}",
            f"numpy=={np.__version__}",
            f"scikit-learn=={sklearn.__version__}",
            f"cloudpickle=={cloudpickle.__version__}"
        ]
    )

    MODEL_URI_CORRIGE = info.model_uri

# TEST CRITIQUE : recharger et prédire réellement
modele_recharge = mlflow.pyfunc.load_model(
    MODEL_URI_CORRIGE
)

pred_recharge = modele_recharge.predict(X_api)

np.testing.assert_allclose(
    pred_recharge,
    pred_reference,
    rtol=1e-8,
    atol=1e-8
)

print("Rechargement MLflow : OK")
print("Prédictions identiques : OK")
print("Model URI corrigé :", MODEL_URI_CORRIGE)
print("Validation cellule 5 : OK")


In [0]:

# CELLULE 6 — Enregistrement dans Unity Catalog

import mlflow
import numpy as np

mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = (
    "dbw_agricoop_poc.gold.prevision_demande_lightgbm"
)

# Modèle corrigé et déjà testé après rechargement
MODEL_URI_CORRIGE = (
    "models:/m-951f26d5d54844aba232fe6330dde0db"
)

# Inscription au registre Unity Catalog
version = mlflow.register_model(
    model_uri=MODEL_URI_CORRIGE,
    name=UC_MODEL_NAME
)

print("Modèle enregistré :", version.name)
print("Version :", version.version)

# Test du modèle depuis le registre
URI_VERSION = f"models:/{UC_MODEL_NAME}/{version.version}"

modele_uc = mlflow.pyfunc.load_model(URI_VERSION)

pred_uc = modele_uc.predict(X_api)

np.testing.assert_allclose(
    pred_uc,
    pred_recharge,
    rtol=1e-8,
    atol=1e-8
)

print("Chargement depuis Unity Catalog : OK")
print("Prédictions identiques : OK")
print("URI modèle :", URI_VERSION)
print("Validation cellule 6 : OK")


In [0]:

# Cellule 7 — Gouvernance du modèle dans Unity Catalog

from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

UC_MODEL_NAME = "dbw_agricoop_poc.gold.prevision_demande_lightgbm"

client.set_registered_model_alias(
    name=UC_MODEL_NAME,
    alias="Candidate",
    version="1"
)

client.update_model_version(
    name=UC_MODEL_NAME,
    version="1",
    description=(
        "AgriCoop Intelligence — LightGBM de prévision "
        "hebdomadaire de la demande. Entraîné sur données "
        "synthétiques sans semaines en rupture. "
        "Validation temporelle sur 52 semaines et "
        "walk-forward avec réentraînement toutes les 4 semaines. "
        "Prototype POC, non validé sur données réelles."
    )
)

version_candidate = client.get_model_version_by_alias(
    UC_MODEL_NAME, "Candidate"
)

print("Modèle :", version_candidate.name)
print("Alias : Candidate")
print("Version :", version_candidate.version)
print("Validation cellule 7 : OK")


In [0]:

# ============================================================
# CELLULE 8 — BATCH SCORING AGriCoop
# Historique walk-forward + simulation après dernier historique
# ============================================================

import pandas as pd
import numpy as np
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = "dbw_agricoop_poc.gold.prevision_demande_lightgbm"
ALIAS = "Candidate"
TABLE = "dbw_agricoop_poc.gold.prevision_demande"

DELTA_PATH = (
    "abfss://gold@stagricoopdata26.dfs.core.windows.net/"
    "prevision_demande_delta/"
)

COOP_ID = "56783517-58a9-42ee-b9d5-c8b946b6b6e3"

# ------------------------------------------------------------
# 1. Charger le modèle PyFunc depuis Unity Catalog
# ------------------------------------------------------------

client = MlflowClient()

modele = mlflow.pyfunc.load_model(
    f"models:/{UC_MODEL_NAME}@{ALIAS}"
)

version = client.get_model_version_by_alias(
    UC_MODEL_NAME, ALIAS
).version

print(f"Modèle chargé : {UC_MODEL_NAME} v{version}")

# ------------------------------------------------------------
# 2. Calendrier religieux du générateur synthétique
# ------------------------------------------------------------

RAMADAN = [
    ("2024-03-11", "2024-04-09"),
    ("2025-03-01", "2025-03-30"),
    ("2026-02-18", "2026-03-19")
]

AID = [
    "2024-04-10", "2025-03-31", "2026-03-20",
    "2024-06-17", "2025-06-07", "2026-05-27"
]

j = pd.Timedelta(days=1)

ramadan_j = {
    d
    for debut, fin in RAMADAN
    for d in pd.date_range(debut, fin)
}

avant_ramadan_j = {
    d
    for debut, _ in RAMADAN
    for d in pd.date_range(
        pd.Timestamp(debut) - 14 * j,
        pd.Timestamp(debut) - j
    )
}

avant_aid_j = {
    d
    for date_aid in AID
    for d in pd.date_range(
        pd.Timestamp(date_aid) - 10 * j,
        pd.Timestamp(date_aid)
    )
}

def compter(semaine, ensemble):
    return sum(
        (semaine + i * j) in ensemble
        for i in range(7)
    )

# ------------------------------------------------------------
# 3. Construire les features après le dernier historique
# ------------------------------------------------------------

derniere_semaine = df["semaine"].max()
prochaine = derniere_semaine + pd.Timedelta(weeks=1)

lignes = []

for code, g in df.sort_values("semaine").groupby(
    "code", observed=True
):
    v = g["ventes"].to_numpy()

    assert len(v) >= 52

    lignes.append({
        "semaine": prochaine,
        "code": str(code),
        "categorie": str(g["categorie"].iloc[-1]),
        "ventes_lag1": v[-1],
        "ventes_lag2": v[-2],
        "ventes_lag4": v[-4],
        "ventes_lag52": v[-52],
        "ventes_moy4": v[-4:].mean()
    })

futur = pd.DataFrame(lignes)

futur["jours_ramadan"] = compter(prochaine, ramadan_j)
futur["jours_avant_ramadan"] = compter(
    prochaine, avant_ramadan_j
)
futur["jours_avant_aid"] = compter(prochaine, avant_aid_j)

futur["semaine_annee"] = int(prochaine.isocalendar().week)
futur["mois"] = (prochaine + 3 * j).month

assert len(futur) == 11

# ------------------------------------------------------------
# 4. Prédire via le modèle PyFunc
# ------------------------------------------------------------

# Le wrapper convertit les catégories en interne.
# MLflow reçoit donc des chaînes et des nombres float64.

X_api = futur[FEATURES_DEMANDE].copy()

for col in ("code", "categorie"):
    X_api[col] = X_api[col].astype(str)

for col in FEATURES_DEMANDE:
    if col not in ("code", "categorie"):
        X_api[col] = pd.to_numeric(
            X_api[col], errors="raise"
        ).astype("float64")

futur["quantite_prevue"] = modele.predict(X_api)

assert futur["quantite_prevue"].notna().all()

# ------------------------------------------------------------
# 5. Historique walk-forward : 572 prévisions
# ------------------------------------------------------------

assert "pred_walk_forward" in test.columns

famille_de = dict(
    zip(df["code"].astype(str), df["famille"])
)

maintenant = pd.Timestamp.now(
    tz="UTC"
).tz_localize(None)

hist = pd.DataFrame({
    "semaine": test["semaine"],
    "code_produit": test["code"].astype(str),
    "quantite_prevue": test["pred_walk_forward"],
    "type_prevision": "historique_walk_forward",
    "version_modele": (
        "walk-forward, réentraîné toutes les 4 semaines"
    )
})

# ------------------------------------------------------------
# 6. Simulation pour la semaine après le dernier historique
# ------------------------------------------------------------

oper = pd.DataFrame({
    "semaine": futur["semaine"],
    "code_produit": futur["code"].astype(str),
    "quantite_prevue": futur["quantite_prevue"],
    "type_prevision": "simulation_apres_historique",
    "version_modele": (
        f"{UC_MODEL_NAME}/{version}@{ALIAS}"
    )
})

# ------------------------------------------------------------
# 7. Consolider dans Gold
# ------------------------------------------------------------

gold = pd.concat([hist, oper], ignore_index=True)

gold["coop_id"] = COOP_ID
gold["famille"] = gold["code_produit"].map(famille_de)

gold["quantite_prevue"] = (
    gold["quantite_prevue"]
    .clip(lower=0)
    .round(2)
)

gold["semaine"] = pd.to_datetime(
    gold["semaine"]
).dt.date

gold["date_calcul"] = maintenant

assert (len(hist), len(oper)) == (572, 11)
assert len(gold) == 583
assert gold["famille"].notna().all()
assert gold["quantite_prevue"].notna().all()

# ------------------------------------------------------------
# 8. Écrire la table Delta EXTERNE dans ADLS Gold
# ------------------------------------------------------------

colonnes_gold = [
    "coop_id",
    "code_produit",
    "famille",
    "semaine",
    "quantite_prevue",
    "type_prevision",
    "version_modele",
    "date_calcul"
]

spark.createDataFrame(gold[colonnes_gold]) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", DELTA_PATH) \
    .saveAsTable(TABLE)

# ------------------------------------------------------------
# 9. Contrôles finaux
# ------------------------------------------------------------

print("\n=== VALIDATION BATCH SCORING ===")
print("Version modèle :", version)
print("Dernière semaine historique :", derniere_semaine.date())
print("Semaine simulée :", prochaine.date())

print("Historique walk-forward :", len(hist))
print("Simulation :", len(oper))
print("Total Gold :", spark.table(TABLE).count())

alimentaire = oper.loc[
    oper["code_produit"].map(famille_de) == "alimentaire",
    "quantite_prevue"
].sum()

print(
    f"Prévision alimentaire : {alimentaire:.1f} unités"
)

display(
    oper[["code_produit", "quantite_prevue"]]
    .round(2)
    .sort_values("code_produit")
)

print("Table Delta :", TABLE)
print("Validation cellule 8 : OK")


In [0]:

# Cellule 9 — Export JSON pour le write-back PostgreSQL

import json
from datetime import datetime, timezone
from pyspark.sql import functions as F

TABLE = "dbw_agricoop_poc.gold.prevision_demande"

# Sélectionner uniquement les 11 simulations
lignes = (
    spark.table(TABLE)
    .filter(F.col("type_prevision") == "simulation_apres_historique")
    .orderBy("code_produit")
    .collect()
)

assert len(lignes) == 11, f"Attendu 11, obtenu {len(lignes)}"

semaine = lignes[0]["semaine"].isoformat()

payload = {
    "schema_version": 1,
    "type_prevision": "simulation_apres_historique",
    "semaine": semaine,
    "exporte_le_utc": datetime.now(timezone.utc).isoformat(),
    "previsions": [
        {
            "coop_id": r["coop_id"],
            "code_produit": r["code_produit"],
            "famille": r["famille"],
            "quantite_prevue": float(r["quantite_prevue"]),
            "version_modele": r["version_modele"]
        }
        for r in lignes
    ]
}

CHEMIN = (
    "abfss://gold@stagricoopdata26.dfs.core.windows.net/"
    f"exports/prevision_produit/{semaine}/previsions.json"
)

dbutils.fs.put(
    CHEMIN,
    json.dumps(payload, ensure_ascii=False, indent=2),
    overwrite=True
)

# Vérification
controle = json.loads(dbutils.fs.head(CHEMIN, 100000))
assert len(controle["previsions"]) == 11

print("Export Gold : OK")
print("Semaine :", semaine)
print("Prévisions :", len(controle["previsions"]))
print("Chemin :", CHEMIN)
